# 05 - Linear mixed models for phase effects

**Purpose**: per-feature test of whether kinematic measurements differ
across phases, accounting for:
- within-subject correlation (subject random intercept)
- within-session correlation (nested session random intercept)
- reach-level noise (residual)

**Input**: ``data.AKDdf`` (raw reaches) restricted to contacted reaches
and the analyzable phase set.

**Output**: three FDR-adjusted results tables, one per analysis:
- Omnibus 4-phase test
- Deficit (Baseline -> Post_Injury_2-4)
- Recovery (Post_Injury_2-4 -> Post_Rehab_Test)

**Small-sample caveat**: statsmodels uses a chi-square Wald
approximation, not Satterthwaite/Kenward-Roger. At small N subjects this
is mildly anti-conservative. Results here are suggestive of direction
and structure; revisit when N grows.


In [ ]:
# parameters
FEATURE_LIST = None   # None -> use helpers.kinematics.get_kinematic_cols(AKDdf) (deduped list)
TOP_N_FIGURE = 20     # How many features to show on the -log10(p) bar chart
FIGSIZE_BARS = (14, 16)


In [ ]:
import numpy as np                                                                # numerical arrays
import pandas as pd                                                               # dataframes
import matplotlib.pyplot as plt                                                   # plotting

from endpoint_ck_analysis.config import ANALYZABLE_PHASES, EXAMPLE_OUTPUT_DIR, CACHE_DIR, FDR_ALPHA # phase set + output paths + FDR alpha for multiple-testing correction
from endpoint_ck_analysis.data_loader import load_all                                              # one-shot loader
from endpoint_ck_analysis.helpers.kinematics import get_kinematic_cols                             # returns the deduplicated list of kinematic feature column names
from endpoint_ck_analysis.helpers.models import run_phase_lmm_for_features                          # fits an LMM per feature, returns FDR-corrected results
from endpoint_ck_analysis.helpers.plotting import stamp_version                                     # figure version footer

EXAMPLE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)                             # ensure output folder exists
data = load_all()                                                                  # uses cache from 00_setup


## 1. Prepare the reach-level dataframe

One row per contacted reach. Phase is an ordered Categorical with
Baseline as the reference level, so every fitted coefficient reads as
"phase X - Baseline".


In [ ]:
contacted = data.AKDdf[data.AKDdf['contact_group'] == 'contacted'].copy()        # only reaches that touched the pellet (uncontacted reaches lack meaningful kinematics); .copy() avoids a SettingWithCopyWarning when we mutate columns below
contacted['phase_group'] = pd.Categorical(                                       # convert phase_group to an ordered Categorical...
    contacted['phase_group'],
    categories=list(ANALYZABLE_PHASES),                                          # ...explicit category order so Baseline is the reference level...
    ordered=True,                                                                # ...ordered=True so contrasts read as "later phase vs Baseline"
)

features = FEATURE_LIST or get_kinematic_cols(contacted)                         # FEATURE_LIST is a parameter override; if None/empty, fall back to the deduplicated kinematic-feature list
print(f'Features to test: {len(features)}')                                      # status; this is the family size FDR will correct over


## 2. Omnibus across all four phases


In [ ]:
omnibus = run_phase_lmm_for_features(contacted, features, fdr_alpha=FDR_ALPHA)               # one LMM per feature, all four phases included; helper internally FDR-corrects across the feature family
print(omnibus.head(15)[['feature', 'phase_p', 'phase_p_adj', 'n_reaches', 'n_subjects', 'converged']]) # show the 15 most-significant features (sorted by p_adj inside the helper); column subset keeps the print readable
omnibus.to_parquet(CACHE_DIR / 'lmm_omnibus.parquet', index=False)                            # cache the full results table for downstream notebooks / re-analysis


## 3. Deficit delta (Baseline vs Post_Injury_2-4)

Same model structure but restricted to two phases, so the phase
coefficient IS the delta.


In [ ]:
deficit_df = contacted[contacted['phase_group'].isin(['Baseline', 'Post_Injury_2-4'])].copy() # restrict to the two-phase deficit comparison so the LMM coefficient IS the delta
deficit_df['phase_group'] = pd.Categorical(deficit_df['phase_group'], categories=['Baseline', 'Post_Injury_2-4']) # rebuild the Categorical with just these two levels; ordering ensures Baseline is the reference
deficit = run_phase_lmm_for_features(deficit_df, features, fdr_alpha=FDR_ALPHA)               # same per-feature LMM as the omnibus, but on the restricted dataframe
print(deficit.head(15)[['feature', 'phase_p', 'phase_p_adj', 'n_reaches', 'n_subjects', 'converged']])
deficit.to_parquet(CACHE_DIR / 'lmm_deficit.parquet', index=False)


## 4. Recovery delta (Post_Injury_2-4 vs Post_Rehab_Test)


In [ ]:
recovery_df = contacted[contacted['phase_group'].isin(['Post_Injury_2-4', 'Post_Rehab_Test'])].copy() # two-phase recovery comparison
recovery_df['phase_group'] = pd.Categorical(recovery_df['phase_group'], categories=['Post_Injury_2-4', 'Post_Rehab_Test']) # Post_Injury_2-4 is the reference here; coefficient reads as "Post_Rehab - Post_Injury"
recovery = run_phase_lmm_for_features(recovery_df, features, fdr_alpha=FDR_ALPHA)
print(recovery.head(15)[['feature', 'phase_p', 'phase_p_adj', 'n_reaches', 'n_subjects', 'converged']])
recovery.to_parquet(CACHE_DIR / 'lmm_recovery.parquet', index=False)


## 5. -log10(adjusted p) summary

Three stacked bar charts -- one per analysis -- of the top features by
significance. Red dashed line marks the FDR cutoff.


In [ ]:
combined = pd.concat([                                                                       # stack the top features from all three analyses with an 'analysis' label column for groupby
    omnibus.head(TOP_N_FIGURE).assign(analysis='Omnibus (all phases)'),                       # .assign() adds a constant-valued column to the dataframe; .head(N) takes the top N rows already sorted by p_adj
    deficit.head(TOP_N_FIGURE).assign(analysis='Deficit (Baseline -> Post_Injury_2-4)'),
    recovery.head(TOP_N_FIGURE).assign(analysis='Recovery (Post_Injury_2-4 -> Post_Rehab_Test)'),
])
combined['neg_log_p'] = -np.log10(combined['phase_p_adj'].clip(lower=1e-30))                  # -log10 inflates small p-values into tall bars; .clip(lower=1e-30) avoids -log10(0)=inf when p_adj rounds to zero

sig_threshold = -np.log10(FDR_ALPHA)                                                          # vertical reference line for the conventional 0.05 cutoff
fig, axes = plt.subplots(3, 1, figsize=FIGSIZE_BARS)                                          # 3 stacked panels (one per analysis)
for ax, (label, group) in zip(axes, combined.groupby('analysis', sort=False)):                # iterate panels and groups in lockstep; sort=False preserves insertion order (omnibus -> deficit -> recovery)
    plot_df = group.dropna(subset=['neg_log_p']).sort_values('neg_log_p')                     # drop features that didn't fit (NaN p_adj) and sort ascending so the strongest bar ends up on top of the horizontal chart
    colors = ['steelblue' if v >= sig_threshold else 'lightgray' for v in plot_df['neg_log_p']] # list comprehension: significant bars get a bold color, non-significant ones gray
    ax.barh(plot_df['feature'], plot_df['neg_log_p'], color=colors)                           # horizontal bars with feature names on y-axis
    ax.axvline(sig_threshold, color='red', linestyle='--', linewidth=1, label=f'FDR q={FDR_ALPHA}') # red dashed line marking the FDR cutoff
    ax.set_xlabel('-log10(FDR-adjusted p)')
    ax.set_title(label)
    ax.legend(loc='lower right')
plt.tight_layout()
stamp_version(fig, label='05 LMM summary')
plt.savefig(EXAMPLE_OUTPUT_DIR / '05_lmm_summary.png', dpi=150, bbox_inches='tight')
plt.show()


<!-- INTERPRETATION_BLOCK -->
## How to read this notebook's output

<details>
<summary>What the per-feature LMM bar chart tells you (click to expand)</summary>

**What the LMM is asking.** For each kinematic feature, the LMM tests
whether the feature value differs across experimental phases (Baseline,
Post_Injury_1, Post_Injury_2-4, Post_Rehab_Test) once subject-level and
session-level variability are accounted for. A surviving feature is one
where the cohort, on average, reaches differently at different phases
-- it's an aspect of motor performance that injury and/or ABT shifted.
Features that don't survive are stable across the experiment regardless
of injury state.

**Three stacked bar charts.** Omnibus phase effect, deficit delta
(Baseline vs Post_Injury_2-4), recovery delta (Post_Injury_2-4 vs
Post_Rehab_Test). Each plots `-log10(FDR-adjusted p)` per feature.

- Bars to the right of the red dashed line (`-log10(0.05)`) = features
  significantly affected by phase after multiple-testing correction.
- A long blue (significant) tail = many kinematic features change
  across phases; reaching is broadly affected by injury and/or ABT.
- Few or no bars past the red line = no detectable phase effects after
  FDR; either the cohort is too small to find them or the effects are
  small relative to within-subject variability.

**Per-analysis interpretation.**

- *Omnibus*: "is anything different across the four phases at all?".
  This is the broadest test; surviving features here are robustly
  phase-dependent. *Example:* if `peak_velocity_mm_per_sec` survives
  the omnibus, the cohort's reaching speed is not stable across the
  experiment; injury and/or post-injury time and/or ABT changed it.
- *Deficit*: "which features dropped between Baseline and
  Post_Injury_2-4?". Surviving features = injury-affected aspects of
  reaching. These are the kinematic axes the C5 contusion shifted.
- *Recovery*: "which features came back between Post_Injury_2-4 and
  Post_Rehab_Test?". Surviving features = ABT-responsive aspects of
  reaching. These are the kinematic axes that improved with training.

**A feature surviving deficit AND recovery** is a strong candidate for
the paper's headline narrative -- injury reliably perturbed it AND ABT
reliably restored some of it. A feature surviving deficit but NOT
recovery dropped at injury and stayed dropped (no ABT response). A
feature surviving recovery but NOT deficit improved with ABT but didn't
clearly drop at injury (could be late-phase task-learning rather than
injury-specific recovery).

**Top vs low -log10(p) -- what they mean.** A feature with a tall bar
on a given test reliably changed across that contrast. A feature with
a near-zero bar didn't change measurably -- mice on average reach the
same way on that feature before and after. *Example:* if
`reach_duration_sec` has a tall bar on the deficit test but a near-zero
bar on the recovery test, mice slowed down post-injury and stayed
slow despite ABT. That's actually informative -- it points to a
deficit aspect ABT didn't restore, which is itself a publishable
finding.

**At small cohort sizes expect few features to clear FDR.** The LMM
uses reach-level data (hundreds of reaches per subject) which gives
within-subject precision, but the inferential N is still the matched-
subject count. The chi-square Wald approximation at small N means raw
p-values are slightly too small; FDR correction partially compensates.

</details>
